# atomic_gdn2 — kernel speedup reproduction

Reproduces the headline scaling numbers of `atomic_gdn2` v0.2.0 on a single
TPU v5e-1 (also runs on TPU v5e-8, Colab TPU).

**Expected output** (see repo [README](../README.md)):

| H | L=2048 | L=4096 |
|---|--------|--------|
| 2 | 193×   | 207×   |
| 4 | 300×   | 322×   |
| 6 | **468×** | **448×** |

Compared to JAX `associative_scan` baseline, kernel fwd+bwd, B=4, D=128.

**Runtime:** ~15–20 minutes on TPU v5e-1.

If your numbers differ by more than ~10%, check the `Environment` cell output
(JAX version, backend, device count). The reference was measured with
JAX 0.11.2 on a single v5e chip.


In [ ]:
# Install atomic-ops from GitHub (clone only, no pip, no jax upgrade risk)
import os, sys, subprocess

SRC = "/tmp/atomic-ops"
if not os.path.isdir(SRC):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Akseleu-J/atomic-ops.git", SRC], check=True)

if SRC not in sys.path:
    sys.path.insert(0, SRC)

# ---- environment report ----
import jax, jax.numpy as jnp
import numpy as np

try:
    import libtpu; LIBTPU = getattr(libtpu, "__version__", "?")
except Exception:
    LIBTPU = "n/a"

print(f"jax      = {jax.__version__}")
print(f"libtpu   = {LIBTPU}")
print(f"backend  = {jax.default_backend()}")
print(f"devices  = {jax.device_count()}")
print(f"python   = {sys.version.split()[0]}")

assert jax.default_backend() == "tpu", \
    f"This notebook requires a TPU backend, got {jax.default_backend()!r}"

REPO_COMMIT = subprocess.run(
    ["git", "-C", SRC, "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip()
print(f"repo     = {REPO_COMMIT}")


In [ ]:
# OLD baseline: JAX lax.associative_scan (verbatim, self-contained).
from functools import partial
import jax, jax.numpy as jnp


def _gdn2_recurrence_impl(k, ea, z, alpha, q, chunk_size, num_chunks,
                          b, n_heads, d_head, dtype):
    def _combine(s1, s2):
        m1, c1 = s1; m2, c2 = s2
        m_new = m2 @ m1
        fro = jnp.sqrt(jnp.sum(jnp.square(m_new), axis=(-2, -1), keepdims=True))
        m_new = m_new * jnp.minimum(1.0, 1.0 / (fro + 1e-6))
        c_new = jnp.nan_to_num(m2 @ c1 + c2, nan=0.0, posinf=1e4, neginf=-1e4)
        return m_new, c_new

    def _to_chunks(t):
        t = t.reshape(b, num_chunks, chunk_size, n_heads, d_head)
        return jnp.moveaxis(t, 1, 0)

    k_ch, ea_ch, z_ch, alpha_ch, q_ch = map(_to_chunks, (k, ea, z, alpha, q))
    eye_bh = jnp.broadcast_to(jnp.eye(d_head, dtype=dtype), (b, n_heads, d_head, d_head))
    zero_bh = jnp.zeros((b, n_heads, d_head, d_head), dtype=dtype)

    def _chunk_step(carry, ci):
        cM, cS = carry; k_c, ea_c, z_c, alpha_c, q_c = ci
        eye = jnp.eye(d_head, dtype=dtype)[None, None, None, :, :]
        M_c = eye * alpha_c[:, :, :, None, :] - k_c[:, :, :, :, None] @ ea_c[:, :, :, None, :]
        C_c = k_c[:, :, :, :, None] @ z_c[:, :, :, None, :]
        P_, S_ = jax.lax.associative_scan(_combine, (M_c, C_c), axis=1)
        gM = jnp.einsum("bchmn,bhnp->bchmp", P_, cM)
        gS = jnp.einsum("bchmn,bhnp->bchmp", P_, cS) + S_
        gS = jnp.nan_to_num(gS, nan=0.0, posinf=1e4, neginf=-1e4)
        out_c = jnp.einsum("bchij,bchi->bchj", gS, q_c)
        return (gM[:, -1], gS[:, -1]), out_c

    _chunk_step = jax.checkpoint(_chunk_step)
    _, out = jax.lax.scan(_chunk_step, (eye_bh, zero_bh),
                          (k_ch, ea_ch, z_ch, alpha_ch, q_ch))
    return jnp.moveaxis(out, 0, 1).reshape(b, num_chunks * chunk_size, n_heads * d_head)


@partial(jax.custom_vjp, nondiff_argnums=(5, 6, 7, 8, 9, 10))
def gdn2_recurrence_safe(k, ea, z, alpha, q, chunk_size, num_chunks,
                         b, n_heads, d_head, dtype):
    return _gdn2_recurrence_impl(k, ea, z, alpha, q, chunk_size, num_chunks,
                                 b, n_heads, d_head, dtype)


def _fwd(k, ea, z, alpha, q, chunk_size, num_chunks, b, n_heads, d_head, dtype):
    primal = jax.checkpoint(lambda k_, ea_, z_, a_, q_: _gdn2_recurrence_impl(
        k_, ea_, z_, a_, q_, chunk_size, num_chunks, b, n_heads, d_head, dtype))
    out, vjp = jax.vjp(primal, k, ea, z, alpha, q)
    return out, vjp


def _bwd(chunk_size, num_chunks, b, n_heads, d_head, dtype, vjp_fn, g):
    grads = vjp_fn(g)
    return tuple(jnp.nan_to_num(jnp.clip(gr, -1e3, 1e3), nan=0.0) for gr in grads)


gdn2_recurrence_safe.defvjp(_fwd, _bwd)


def fwd_old(q, k, v, w, b, g, scale, h0):
    B, L, H, D = q.shape
    alpha = jnp.exp(g.astype(jnp.float32))
    ea = ((b * k) * alpha).astype(q.dtype)
    z = (w * v).astype(q.dtype)
    num_chunks = L // 128
    out = gdn2_recurrence_safe(k, ea, z, alpha.astype(q.dtype), q,
                               128, num_chunks, B, H, D, q.dtype)
    out = out.reshape(B, L, H, D)
    return out, jnp.zeros((B, H, D, D), dtype=q.dtype)


print("[ok] OLD (associative_scan) defined")


In [ ]:
# NEW: atomic_gdn2 (Pallas). PROD: atomic_ops fallback (optional).
import math

from atomic_gdn2 import make_cfg, make_blr_trainable

CFG_NEW = make_cfg(bt=128, bc=64, score_bs=128, bs2=64, mb=8)
try:
    CFG_NEW = CFG_NEW.with_(b3_dot_mode="bf16x3", b4_dot_mode="bf16x3", diag="btl")
except Exception:
    pass

_new_fn = make_blr_trainable(CFG_NEW, 1.0 / math.sqrt(128))

def fwd_new(q, k, v, w, b, g, scale, h0):
    sw = lambda t: jnp.swapaxes(t, 1, 2)
    o_hm, hf = _new_fn(sw(q), sw(k), sw(v), sw(w), sw(b), sw(g), h0)
    return sw(o_hm), hf

print("[ok] NEW (atomic_gdn2 Pallas) ready")


fwd_prod = None
try:
    from atomic_ops.fallback import gdn2_forward_trainable as _PROD_FN
    from atomic_ops.configs import KernelConfig as _KC
    _PROD_KCFG = _KC(bt=128, bc=64, mb=8, wy_eps=1e-3)

    def fwd_prod(q, k, v, w, b, g, scale, h0):
        out = _PROD_FN(q, k, v, w, b, g, scale, h0=h0, config=_PROD_KCFG)
        return (out[0], out[1]) if isinstance(out, tuple) else \
               (out, jnp.zeros((q.shape[0], q.shape[2], q.shape[3], q.shape[3]), jnp.float32))

    print("[ok] PROD (atomic_ops fallback) ready")
except Exception as e:
    print(f"[skip] PROD unavailable: {type(e).__name__}")


In [ ]:
# Benchmark helpers
import time, gc
import numpy as np
import jax, jax.numpy as jnp

BENCH_B = 4
BENCH_D = 128
SCALE_K = 1.0 / math.sqrt(BENCH_D)

CONFIGS = [
    {"H": 2, "L": 2048}, {"H": 2, "L": 4096},
    {"H": 4, "L": 2048}, {"H": 4, "L": 4096},
    {"H": 6, "L": 2048}, {"H": 6, "L": 4096},
]


def make_inputs(seed, B, L, H, D):
    key = jax.random.PRNGKey(seed)
    k1, k2, k3, k4, k5, k6, k7 = jax.random.split(key, 7)
    eps = 1e-6
    q = jax.random.normal(k1, (B, L, H, D)) * 0.3
    k = jax.random.normal(k2, (B, L, H, D)) * 0.3
    q = q / (jnp.linalg.norm(q, axis=-1, keepdims=True) + eps)
    k = k / (jnp.linalg.norm(k, axis=-1, keepdims=True) + eps)
    v = jax.random.normal(k3, (B, L, H, D)) * 0.3
    g = -jnp.abs(jax.random.normal(k4, (B, L, H, D))) * 0.05
    b = jax.random.uniform(k5, (B, L, H, D), minval=0.2, maxval=1.0)
    w = jax.random.uniform(k6, (B, L, H, D), minval=0.2, maxval=1.0)
    h0 = jax.random.normal(k7, (B, H, D, D)) * 0.1
    return q, k, v, w, b, g, h0


def bench_one(fwd, args, n_iters=5, warmup=1):
    q, k, v, w, b, g, h0 = args
    def loss(q_, k_, v_, w_, b_, g_):
        o, hf = fwd(q_, k_, v_, w_, b_, g_, SCALE_K, h0)
        return jnp.sum(o.astype(jnp.float32) ** 2) + jnp.sum(hf.astype(jnp.float32) ** 2)
    gf = jax.jit(jax.grad(loss, argnums=(0, 1, 2, 3, 4, 5)))
    for _ in range(warmup):
        jax.block_until_ready(gf(q, k, v, w, b, g))
    ts = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        jax.block_until_ready(gf(q, k, v, w, b, g))
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)) * 1e3, float(np.std(ts)) * 1e3


print("[ok] helpers ready")


In [ ]:
# Run benchmark: H in {2,4,6} x L in {2048,4096}
paths = {"OLD": fwd_old, "NEW": fwd_new}
if fwd_prod is not None:
    paths["PROD"] = fwd_prod

results = {}
for cfg in CONFIGS:
    H, L = cfg["H"], cfg["L"]
    key = f"H{H}_L{L}"
    print(f"\n=== {key}  B={BENCH_B} H={H} L={L} D={BENCH_D} ===")
    args = make_inputs(0, BENCH_B, L, H, BENCH_D)
    results[key] = {}
    for name, fwd in paths.items():
        try:
            o_test, h_test = fwd(*args, SCALE_K, args[-1])
            is_fin = bool(jnp.all(jnp.isfinite(o_test))) and bool(jnp.all(jnp.isfinite(h_test)))
            del o_test, h_test
            gc.collect()
            try: jax.clear_caches()
            except Exception: pass
            if not is_fin:
                print(f"  {name:5s}  NON-FINITE — skip"); results[key][name] = None; continue
            n_iter = 3 if name == "OLD" else 5
            ms, sd = bench_one(fwd, args, n_iters=n_iter, warmup=1)
            results[key][name] = ms
            print(f"  {name:5s}  {ms:10.3f} ms  (std {sd:.3f})")
        except Exception as e:
            print(f"  {name:5s}  FAIL: {type(e).__name__}: {str(e)[:200]}")
            results[key][name] = None
        finally:
            gc.collect()
            try: jax.clear_caches()
            except Exception: pass


In [ ]:
# Correctness check: NEW vs JAX_REF at H=4,6 / L=2048
try:
    from atomic_gdn2.reference import forward_ref as _REF
    CFG_REF = make_cfg(bt=128, bc=64, score_bs=128, bs2=64, mb=8)
    try:
        CFG_REF = CFG_REF.with_(b3_dot_mode="bf16x3", b4_dot_mode="bf16x3", diag="btl")
    except Exception:
        pass

    def fwd_ref(q, k, v, w, b, g, scale, h0):
        r = _REF(q, k, v, w, b, g, scale, h0, CFG_REF)
        if isinstance(r, tuple): return r[0], r[1]
        return r, jnp.zeros((q.shape[0], q.shape[2], q.shape[3], q.shape[3]), jnp.float32)

    def rel_l2(a, b):
        a = np.asarray(jax.device_get(a), np.float64)
        b = np.asarray(jax.device_get(b), np.float64)
        return float(np.linalg.norm((a - b).ravel()) / (np.linalg.norm(b.ravel()) + 1e-12))

    print("Correctness: rel_l2(NEW, JAX_REF)")
    for (H, L) in [(4, 2048), (6, 2048)]:
        args = make_inputs(0, BENCH_B, L, H, BENCH_D)
        o_ref, _ = fwd_ref(*args, SCALE_K, args[-1])
        o_new, _ = fwd_new(*args, SCALE_K, args[-1])
        rel = rel_l2(o_new, o_ref)
        verdict = "OK" if rel < 5e-2 else "WARN"
        print(f"  H={H} L={L}: rel_l2 = {rel:.3e}  [{verdict}]")
        del o_ref, o_new, args
        gc.collect()
        try: jax.clear_caches()
        except Exception: pass
except Exception as e:
    print(f"[skip] correctness check unavailable: {type(e).__name__}: {e}")


In [ ]:
# Summary
import json

print("\n" + "=" * 60)
print("atomic_gdn2 kernel speedup vs associative_scan")
print(f"  B={BENCH_B} D={BENCH_D}  backend={jax.default_backend()}  "
      f"devices={jax.device_count()}")
print("=" * 60)
print(f"  {'H':<4} {'L=2048':>12} {'L=4096':>12}")
print("  " + "-" * 32)

rows = []
for H in (2, 4, 6):
    cells = []
    for L in (2048, 4096):
        key = f"H{H}_L{L}"
        old_ms = results.get(key, {}).get("OLD")
        new_ms = results.get(key, {}).get("NEW")
        sp = old_ms / new_ms if (old_ms and new_ms) else None
        cells.append(sp)
        if sp is not None:
            rows.append({"H": H, "L": L, "OLD_ms": old_ms, "NEW_ms": new_ms, "speedup": sp})
    s1 = f"{cells[0]:>10.2f}x" if cells[0] else f"{'n/a':>11}"
    s2 = f"{cells[1]:>10.2f}x" if cells[1] else f"{'n/a':>11}"
    print(f"  {H:<4} {s1} {s2}")

print("\nExpected (repo README): 193x / 207x / 300x / 322x / 468x / 448x")

# Save JSON
out = {
    "jax": jax.__version__,
    "backend": jax.default_backend(),
    "devices": jax.device_count(),
    "B": BENCH_B, "D": BENCH_D,
    "configs": rows,
}
with open("speedup_scaling_output.json", "w") as f:
    json.dump(out, f, indent=2)
print("\n[ok] saved speedup_scaling_output.json")


## How to read this

- **OLD** = JAX `lax.associative_scan`, the standard reference implementation.
- **NEW** = `atomic_gdn2` fused Pallas kernel (`make_blr_trainable`).
- **Speedup** = `OLD_ms / NEW_ms`, kernel fwd+bwd only.

Speedup grows with head count H because `associative_scan` materializes an
O(H·L·d²) prefix-scan state, while `atomic_gdn2` is O(L·d²). On a single chip
the gain peaks at H=6; beyond that the fused kernel's tile pack stops fitting
cleanly and sharding across chips (TPU v5e-8) is required for further scaling.

If your numbers differ substantially from the reference table, compare the
`Environment` cell output with the version pinned in
[`attestation/scaling.json`](../attestation/scaling.json).
